<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.2-channel-flow/Ex09.2_03_shape_comparison.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.2 · Notebook 03 — comparing shapes fairly

**Paired with L9.2 · Turbulent Flow**

A drag comparison is only meaningful if everything else is equal. Match the
**blockage ratio**, not the nominal size: a diamond of size 0.2 blocks a
different fraction of the channel than an ellipse of size 0.2.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.2-channel-flow/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
# --- paste your residual_fn and loss_fn_factory from notebook 01 here ---
raise NotImplementedError("paste your residual_fn and loss_fn_factory")

## 1 · TODO — run every shape at matched blockage and Reynolds number

In [ ]:
RE = 100.0
TARGET_BLOCKAGE = 0.4

shapes_run = []
for sh in pb.SHAPES:
    size = TARGET_BLOCKAGE * pb.CHANNEL["H"] / 2.0
    cfg = pb.PipeConfig(shape=sh, size=size, reynolds=RE,
                        n_collocation=8000, adam_epochs=3000, lbfgs_epochs=300)
    print(f"\n=== {sh}  (blockage {cfg.blockage:.2f}) ===")
    shapes_run.append(pb.run_case(cfg, residual_fn, loss_fn_factory, verbose=False))
    r = shapes_run[-1]
    print(f"  C_D {r['C_D']:.3f}   dp {r['dp']:.4f}   loss {r['final_loss']:.2e}")

## 2 · The comparison

In [ ]:
names = [r["config"].shape for r in shapes_run]
cds   = [r["C_D"] for r in shapes_run]
dps   = [r["dp"]  for r in shapes_run]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(names, cds, color=CYCLE[0]); ax[0].set_ylabel("$C_D$")
ax[0].tick_params(axis="x", rotation=20)
ax[1].bar(names, dps, color=CYCLE[1]); ax[1].set_ylabel("pressure drop")
ax[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

print(error_table(
    [[n, f"{c:.3f}", f"{d:.4f}", f"{r['final_loss']:.2e}"]
     for n, c, d, r in zip(names, cds, dps, shapes_run)],
    ["shape", "C_D", "pressure drop", "final loss"]))

**Interpret before you report.** A streamlined shape should show lower drag
than a bluff one at the same blockage, because the flow stays attached further
back and the wake is narrower. If your ordering disagrees, ask whether the
near-wall sampling is adequate before believing the numbers — drag is the
quantity most sensitive to it.

---

## 3 · Save

In [ ]:
import pickle

os.makedirs("Ex09.2_outputs", exist_ok=True)
path = os.path.join("Ex09.2_outputs", "nb03_shapes.pkl")
with open(path, "wb") as f:
    pickle.dump([{k: v for k, v in r.items() if k != "model"} for r in shapes_run], f)
print("wrote", path)